<a href="https://colab.research.google.com/github/JeysonCarmona/PPMI_INVESTIGATION/blob/main/notebook4_integrated_Patient_Explorer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Notebook 4 — Integrated Patient Explorer (PPMI)

**Objective:** Given a `PATNO`, display in one place their available clinical information, a summary of their imaging studies (with on-demand extraction of only some DICOMs), and a summary of their exome (with on-demand extraction of only their VCF).

**Restrictions observed throughout the notebook:**
- `Imagenes_PPMI.rar` is never fully uncompressed. Only headers are read (`unrar lb`) and, on demand, a few `.dcm` files of the queried patient are extracted (`unrar x` pointing to specific paths).
- The 78 `.tar.gz` files are never fully uncompressed. Only the `.tar.gz` containing the patient is opened, and only their `.raw.vcf` is extracted.

**Note on input data:** `Pacientes__1_.csv` (as uploaded) does not include `PATNO`/`Alias SI`/`Group`/`Participant status` as explicit columns — it's the raw export of IDA image metadata (`Subject ID`, `Sex`, `Age`, `Description`). This notebook assembles the best possible clinical information with what's available and explicitly marks fields depending on `participant_index.csv` (the merged clinical index from Notebook 1) as "not available", as it has not yet been uploaded to this location. As soon as you upload it to the results folder, the notebook automatically detects and uses it (see Section 1).

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!apt-get install -y unrar -qq
!pip install pydicom -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 37.4 MB/s eta 0:00:00


## 0. Path Configuration

Same directories used in notebooks 1-3.

In [3]:
import os

BASE_DIR = "/content/drive/MyDrive/Investigación_Parkinson"
RAR_IMAGENES = BASE_DIR + "/Imagenes/Imagenes_PPMI.rar"
DIR_EXOMAS = BASE_DIR + "/Exomas - Parkinson"
RESULTADOS_DIR = BASE_DIR + "/Jeyson_Carmona_Michael_Lamprea/segunda entrega/Resultados"
DIR_CONTEO = BASE_DIR + "/Conteo pacientes"

# Working folders ONLY for the few files extracted on demand
# (never for the full RAR or TARs). They are cleaned at the end of the notebook.
DICOM_TEMP_DIR = "/content/dicom_temp"
VCF_TEMP_DIR = "/content/vcf_temp"
os.makedirs(DICOM_TEMP_DIR, exist_ok=True)
os.makedirs(VCF_TEMP_DIR, exist_ok=True)

# Paths to the indices already generated in notebooks 1-3
RUTA_IMAGENES_INDEX = RESULTADOS_DIR + "/imagenes_index.csv"
RUTA_EXOMAS_INDEX = RESULTADOS_DIR + "/exomas_index.csv"
RUTA_PARTICIPANT_INDEX = RESULTADOS_DIR + "/participant_index.csv"   # ideal, from Notebook 1
RUTA_PACIENTES_CSV = DIR_CONTEO + "/Pacientes.csv"                    # the one we have (Subject ID, Sex, Age, Description)
RUTA_CACHE_RUTAS_DICOM = RESULTADOS_DIR + "/rutas_dicom_completas.csv"  # built in Section 2

for r in [RAR_IMAGENES, DIR_EXOMAS, RUTA_IMAGENES_INDEX, RUTA_EXOMAS_INDEX]:
    assert os.path.exists(r), f"Not found: {r}"
print("Base paths verified OK.")

Base paths verified OK.


## 1. Clinical Information

Assembled with available data:
- `diagnosis_group` (Control / Parkinson_Disease) from `imagenes_index.csv`.
- `Sex` / `Age` from the raw patient CSV (if available).
- `Alias SI`, `Participant status`, and any other clinical field: taken from `participant_index.csv` **if it exists** in `Resultados/`; otherwise, they are marked as "not available" without inventing values.

## 1. Información clínica

Se arma con lo disponible:
- `grupo_diagnostico` (Control / Parkinson_Disease) desde `imagenes_index.csv`.
- `Sex` / `Age` desde el CSV crudo de pacientes (si está disponible).
- `Alias SI`, `Estado del participante` y cualquier otro campo clínico: se toman de `participant_index.csv` **si existe** en `Resultados/`; si no, se marcan como *"no disponible"* sin inventar valores.

In [4]:
import pandas as pd

df_imagenes_idx = pd.read_csv(RUTA_IMAGENES_INDEX)
df_exomas_idx = pd.read_csv(RUTA_EXOMAS_INDEX)
df_exomas_idx["PATNO"] = df_exomas_idx["PATNO"].astype(int)

df_pacientes_raw = None
if os.path.exists(RUTA_PACIENTES_CSV):
    df_pacientes_raw = pd.read_csv(RUTA_PACIENTES_CSV)

df_participant_index = None
if os.path.exists(RUTA_PARTICIPANT_INDEX):
    df_participant_index = pd.read_csv(RUTA_PARTICIPANT_INDEX)
    print("participant_index.csv found: will be used for Alias SI / Group / Status.")
else:
    print("participant_index.csv NOT found in Results/. "
          "Alias SI and Participant status will be shown as 'not available' "
          "until you upload that file (generated by Notebook 1).")

participant_index.csv found: will be used for Alias SI / Group / Status.


In [5]:
def buscar_info_clinica(patno):
    """Assembles a dictionary with available clinical information for a PATNO.
    Does not invent values: if a field is not in any source, it is explicitly marked.
    """
    patno = int(patno)
    info = {
        "PATNO": patno,
        "Alias_SI": "not available",
        "Sexo": "not available",
        "Edad": "not available",
        "Grupo": "not available",
        "Estado_participante": "not available",
        "Fuente": [],
    }

    # 1) participant_index.csv (ideal source, from Notebook 1)
    if df_participant_index is not None:
        fila = df_participant_index[df_participant_index.get("PATNO", pd.Series(dtype=int)) == patno]
        if not fila.empty:
            fila = fila.iloc[0]
            for campo_destino, posibles_columnas in [
                ("Alias_SI", ["Alias_SI", "Alias SI", "SI", "SITE_ID"]),
                ("Sexo", ["Sexo", "Sex", "SEX"]),
                ("Edad", ["Edad", "Age"]),
                ("Grupo", ["Grupo", "COHORT", "Diagnosis", "grupo_diagnostico"]),
                ("Estado_participante", ["Estado_participante", "ENROLL_STATUS", "Estado"]),
            ]:
                for col in posibles_columnas:
                    if col in fila.index and pd.notna(fila[col]):
                        info[campo_destino] = fila[col]
                        break
            info["Fuente"].append("participant_index.csv")

    # 2) Raw patient CSV (Sex, Age) by Subject ID == PATNO
    if info["Sexo"] == "not available" and df_pacientes_raw is not None:
        fila = df_pacientes_raw[df_pacientes_raw["Subject ID"] == patno]
        if not fila.empty:
            info["Sexo"] = fila.iloc[0].get("Sex", "not available")
            info["Edad"] = fila.iloc[0].get("Age", "not available")
            info["Fuente"].append("Pacientes.csv (raw IDA)")

    # 3) Diagnostic group from imagenes_index.csv, if not from participant_index
    if info["Grupo"] == "not available":
        fila_img = df_imagenes_idx[df_imagenes_idx["PATNO"] == patno]
        if not fila_img.empty and fila_img["grupo_diagnostico"].notna().any():
            info["Grupo"] = fila_img["grupo_diagnostico"].dropna().iloc[0]
            info["Fuente"].append("imagenes_index.csv")

    info["Fuente"] = ", ".join(info["Fuente"]) if info["Fuente"] else "no matches in any source"
    return info


def mostrar_info_clinica(patno):
    info = buscar_info_clinica(patno)
    print("=" * 50)
    print(f"CLINICAL INFORMATION — PATNO {info['PATNO']}")
    print("=" * 50)
    print(f"Alias SI:              {info['Alias_SI']}")
    print(f"Sex:                   {info['Sexo']}")
    print(f"Age:                   {info['Edad']}")
    print(f"Group:                 {info['Grupo']}")
    print(f"Participant Status:    {info['Estado_participante']}")
    print(f"Source(s) used:        {info['Fuente']}")
    return info

## 2. Images

`imagenes_index.csv` is aggregated by study (it lost the individual paths of each `.dcm`), so it's not enough to extract specific files. This section builds, **only once**, a cache with the full paths within the RAR (`rutas_dicom_completas.csv`) and reuses it in subsequent executions — so the entire RAR is not rescanned every time you query a patient.

It also corrects the date bug from Notebook 3 (the regex required exact `YYYY-MM-DD`; actual folders include time: `YYYY-MM-DD_HH_MM_SS.s`).

In [6]:
import subprocess
import re

def listar_rutas_rar(ruta_rar, unrar_path="unrar"):
    """Reads only the internal names of the RAR via 'unrar lb -r' (list bare,
    recursive). Only reads headers: does not extract or decompress content."""
    comando = [unrar_path, "lb", "-r", ruta_rar]
    proceso = subprocess.Popen(
        comando, stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True, bufsize=1,
    )
    rutas_archivos = []
    for linea in proceso.stdout:
        linea = linea.rstrip("\n").replace("\\", "/")
        if linea and not linea.endswith("/"):
            rutas_archivos.append(linea)
    proceso.stdout.close()
    codigo_retorno = proceso.wait()
    stderr_out = proceso.stderr.read()
    proceso.stderr.close()
    if codigo_retorno != 0:
        raise RuntimeError(f"unrar returned code {codigo_retorno}. Detail: {stderr_out}")
    return rutas_archivos


PATRON_GRUPO = re.compile(r"(Control|Parkinson_Disease|PD)", re.IGNORECASE)
PATRON_VISTA = re.compile(r"(Axial|Sagital)", re.IGNORECASE)
PATRON_SECUENCIA = re.compile(r"(T1|T2)", re.IGNORECASE)
PATRON_DIMENSION = re.compile(r"(2D|3D)", re.IGNORECASE)
# Correction: accepts date with or without time suffix (YYYY-MM-DD[_HH_MM_SS.s])
PATRON_FECHA = re.compile(r"^(\d{4}-\d{2}-\d{2})")

def parsear_ruta_dicom(ruta):
    partes = ruta.split("/")
    grupo = vista = secuencia = dimension = patno = estudio_desc = fecha = None

    m = PATRON_GRUPO.search(ruta)
    if m: grupo = m.group(1)
    m = PATRON_VISTA.search(ruta)
    if m: vista = m.group(1)
    m = PATRON_SECUENCIA.search(ruta)
    if m: secuencia = m.group(1)
    m = PATRON_DIMENSION.search(ruta)
    if m: dimension = m.group(1)

    for i, parte in enumerate(partes):
        if parte.upper() == "PPMI" and i + 1 < len(partes):
            posible_patno = partes[i + 1]
            if posible_patno.isdigit():
                patno = int(posible_patno)
                if i + 2 < len(partes):
                    estudio_desc = partes[i + 2]
                if i + 3 < len(partes):
                    m_fecha = PATRON_FECHA.match(partes[i + 3])
                    if m_fecha:
                        fecha = m_fecha.group(1)
            break

    return {
        "ruta_completa": ruta, "grupo_diagnostico": grupo, "vista": vista,
        "secuencia": secuencia, "dimension": dimension, "PATNO": patno,
        "estudio": estudio_desc, "fecha": fecha,
        "nombre_archivo": partes[-1] if partes else None,
    }

In [7]:
# Builds the full path cache ONLY ONCE (high cost, minutes/hours
# depending on Drive speed). If it already exists, it is simply loaded from disk.
if os.path.exists(RUTA_CACHE_RUTAS_DICOM):
    df_rutas_dicom = pd.read_csv(RUTA_CACHE_RUTAS_DICOM)
    print(f"Cache loaded from disk: {len(df_rutas_dicom):,} paths.")
else:
    print("Cache does not exist yet. Scanning RAR headers (this may take a while)...")
    rutas = listar_rutas_rar(RAR_IMAGENES)
    print(f"Paths read: {len(rutas):,}. Parsing...")
    registros = [parsear_ruta_dicom(r) for r in rutas]
    df_rutas_dicom = pd.DataFrame(registros)
    df_rutas_dicom.to_csv(RUTA_CACHE_RUTAS_DICOM, index=False)
    print(f"Cache saved to: {RUTA_CACHE_RUTAS_DICOM}")

/tmp/ipykernel_5981/3202371505.py:4: DtypeWarning: Columns (3,6,7) have mixed types. Specify dtype option on import or set low_memory=False.
  df_rutas_dicom = pd.read_csv(RUTA_CACHE_RUTAS_DICOM)


Cache loaded from disk: 3,277,408 paths.


In [8]:
def resumen_imagenes_paciente(patno):
    """Summary by study for a PATNO, using the full path cache."""
    patno = int(patno)
    df_pac = df_rutas_dicom[df_rutas_dicom["PATNO"] == patno]
    if df_pac.empty:
        print(f"No images found for PATNO {patno} within the RAR.")
        return None

    resumen = df_pac.groupby(
        ["estudio", "fecha", "vista", "secuencia", "dimension"], dropna=False
    ).size().reset_index(name="n_imagenes")

    print(f"PATNO {patno} — {resumen.shape[0]} study(ies) found, "
          f"{len(df_pac)} DICOM files in total.")
    display(resumen)
    return df_pac

In [9]:
def extraer_dicom_muestra(patno, n_por_estudio=3):
    """Extracts ONLY a few .dcm per study of the indicated patient, using
    'unrar x' pointing to specific paths within the RAR. Never extracts the full RAR
    or all the patient's files."""
    patno = int(patno)
    df_pac = df_rutas_dicom[df_rutas_dicom["PATNO"] == patno]
    if df_pac.empty:
        print(f"No indexed images for PATNO {patno}.")
        return []

    muestra = (
        df_pac.groupby(["estudio", "fecha"], dropna=False)
        .head(n_por_estudio)
    )

    destino = os.path.join(DICOM_TEMP_DIR, str(patno))
    os.makedirs(destino, exist_ok=True)

    archivos_extraidos = []
    for ruta_interna in muestra["ruta_completa"]:
        comando = ["unrar", "x", "-y", RAR_IMAGENES, ruta_interna, destino + "/"]
        resultado = subprocess.run(comando, capture_output=True, text=True)
        if resultado.returncode == 0:
            nombre = ruta_interna.split("/")[-1]
            ruta_local = os.path.join(destino, nombre)
            if os.path.exists(ruta_local):
                archivos_extraidos.append(ruta_local)
        else:
            print(f"Could not extract {ruta_interna}: {resultado.stderr.strip()}")

    print(f"Extracted {len(archivos_extraidos)} DICOM file(s) in: {destino}")
    return archivos_extraidos

In [10]:
import pydicom
import matplotlib.pyplot as plt

def mostrar_miniaturas_dicom(rutas_locales, max_imagenes=6):
    """Displays a grid of thumbnails from DICOM files already extracted
    locally (never reads directly from within the RAR)."""
    rutas_locales = rutas_locales[:max_imagenes]
    if not rutas_locales:
        print("No local DICOM files to display.")
        return

    n = len(rutas_locales)
    cols = min(3, n)
    filas = (n + cols - 1) // cols
    fig, axes = plt.subplots(filas, cols, figsize=(4 * cols, 4 * filas))
    axes = axes.flatten() if n > 1 else [axes]

    for ax, ruta in zip(axes, rutas_locales):
        try:
            ds = pydicom.dcmread(ruta)
            ax.imshow(ds.pixel_array, cmap="gray")
            ax.set_title(os.path.basename(ruta), fontsize=8)
        except Exception as e:
            ax.set_title(f"Error: {e}", fontsize=8)
        ax.axis("off")

    for ax in axes[len(rutas_locales):]:
        ax.axis("off")

    plt.tight_layout()
    plt.show()

## 3. Exome

`exomas_index.csv` already provides, for each `PATNO`, the `.tar.gz` and the exact name of the `.vcf` within that TAR — no need to rescan anything. The TAR is opened in streaming mode and **only** the member corresponding to the patient is extracted.

In [11]:
import tarfile

def localizar_exoma(patno):
    patno = int(patno)
    fila = df_exomas_idx[df_exomas_idx["PATNO"] == patno]
    if fila.empty:
        print(f"No exome indexed for PATNO {patno}.")
        return None
    fila = fila.iloc[0]
    return {
        "PATNO": patno,
        "archivo_tar": fila["archivo_tar"],
        "archivo_vcf": fila["archivo_vcf"],
        "tamano_bytes": fila["tamano_bytes"],
    }


def extraer_vcf_paciente(patno):
    """Opens ONLY the .tar.gz corresponding to the patient and extracts ONLY its .vcf,
    without touching the other 77 TARs or the other members of the opened TAR."""
    info = localizar_exoma(patno)
    if info is None:
        return None

    ruta_tar = os.path.join(DIR_EXOMAS, info["archivo_tar"])
    if not os.path.exists(ruta_tar):
        print(f"TAR not found on disk: {ruta_tar}")
        return None

    destino = os.path.join(VCF_TEMP_DIR, str(patno))
    os.makedirs(destino, exist_ok=True)
    ruta_local_vcf = os.path.join(destino, info["archivo_vcf"])

    if os.path.exists(ruta_local_vcf):
        print(f"VCF already extracted previously: {ruta_local_vcf}")
        return ruta_local_vcf

    with tarfile.open(ruta_tar, mode="r:gz") as tar:
        miembro = None
        for m in tar:
            if m.name.endswith(info["archivo_vcf"]):
                miembro = m
                break
        if miembro is None:
            print(f"Could not find {info['archivo_vcf']} within {info['archivo_tar']}.")
            return None
        with tar.extractfile(miembro) as origen, open(ruta_local_vcf, "wb") as salida:
            while True:
                bloque = origen.read(1024 * 1024)
                if not bloque:
                    break
                salida.write(bloque)

    print(f"VCF extracted to: {ruta_local_vcf}")
    return ruta_local_vcf

In [12]:
# Genes commonly associated with Parkinson's, used only to highlight them if
# mentioned in the INFO field (e.g., ANN=/CSQ= annotation from VEP/SnpEff).
# If the VCF is "raw" (without annotation), this list will simply not find matches
# and the summary explicitly indicates this instead of inventing findings.
GENES_PD_RELEVANTES = ["SNCA", "LRRK2", "GBA", "PRKN", "PARK2", "PINK1", "PARK7", "VPS35", "UCHL1"]

def resumen_vcf(ruta_vcf, max_lineas_muestra=200000):
    """Light summary of a VCF by streaming line by line (does not load everything into memory)."""
    n_variantes = 0
    cromosomas = set()
    genes_encontrados = set()
    tiene_anotacion = False
    columnas_header = None

    abrir = open
    if ruta_vcf.endswith(".gz"):
        import gzip
        abrir = gzip.open

    with abrir(ruta_vcf, "rt", errors="replace") as f:
        for i, linea in enumerate(f):
            if linea.startswith("##"):
                continue
            if linea.startswith("#CHROM"):
                columnas_header = linea.strip().split("\t")
                continue
            campos = linea.rstrip("\n").split("\t")
            if len(campos) < 8:
                continue
            n_variantes += 1
            cromosomas.add(campos[0])
            info_field = campos[7]
            if "ANN=" in info_field or "CSQ=" in info_field:
                tiene_anotacion = True
                for gen in GENES_PD_RELEVANTES:
                    if gen in info_field:
                        genes_encontrados.add(gen)
            if i > max_lineas_muestra:
                break

    resumen = {
        "ruta_vcf": ruta_vcf,
        "n_variantes_leidas": n_variantes,
        "cromosomas_presentes": sorted(cromosomas, key=lambda c: (len(c), c)),
        "tiene_anotacion_genica": tiene_anotacion,
        "genes_PD_relevantes_detectados": sorted(genes_encontrados) if tiene_anotacion else "N/A (VCF without genomic annotation)",
    }
    return resumen


def mostrar_resumen_vcf(patno):
    ruta_vcf = extraer_vcf_paciente(patno)
    if ruta_vcf is None:
        return None
    resumen = resumen_vcf(ruta_vcf)
    print("=" * 50)
    print(f"VCF SUMMARY — PATNO {patno}")
    print("=" * 50)
    print(f"Variants read:             {resumen['n_variantes_leidas']:,}")
    print(f"Chromosomes present:       {', '.join(resumen['cromosomas_presentes'])}")
    print(f"Is VCF annotated (ANN/CSQ)?: {resumen['tiene_anotacion_genica']}")
    print(f"Relevant PD genes:         {resumen['genes_PD_relevantes_detectados']}")
    return resumen

## 4. Integrated Explorer

Single function that combines the three sections for a given `PATNO`.

In [13]:
def explorar_paciente(patno, extraer_imagenes=True, extraer_exoma=True, n_dicom_por_estudio=3):
    patno = int(patno)
    print("#" * 60)
    print(f"# PATIENT EXPLORER — PATNO {patno}")
    print("#" * 60)

    print("\n--- 1. CLINICAL INFORMATION ---")
    info_clinica = mostrar_info_clinica(patno)

    print("\n--- 2. IMAGES ---")
    df_pac_img = resumen_imagenes_paciente(patno)
    archivos_dicom = []
    if extraer_imagenes and df_pac_img is not None:
        archivos_dicom = extraer_dicom_muestra(patno, n_por_estudio=n_dicom_por_estudio)
        mostrar_miniaturas_dicom(archivos_dicom)

    print("\n--- 3. EXOME ---")
    resumen_exoma = None
    if extraer_exoma:
        resumen_exoma = mostrar_resumen_vcf(patno)
    else:
        info_exoma = localizar_exoma(patno)

    return {
        "clinica": info_clinica,
        "imagenes": df_pac_img,
        "dicom_extraidos": archivos_dicom,
        "exoma": resumen_exoma,
    }

In [14]:
# Example of use:
# resultado = explorar_paciente(3184)

PATNO_A_EXPLORAR = 3000  # <-- change this value for the patient you want to query
resultado = explorar_paciente(PATNO_A_EXPLORAR)

############################################################
# PATIENT EXPLORER — PATNO 3000
############################################################

--- 1. CLINICAL INFORMATION ---
CLINICAL INFORMATION — PATNO 3000
Alias SI:              not available
Sex:                   not available
Age:                   not available
Group:                 2
Participant Status:    Withdrew
Source(s) used:        participant_index.csv

--- 2. IMAGES ---
PATNO 3000 — 6 study(ies) found, 172 DICOM files in total.


,estudio,fecha,vista,secuencia,dimension,n_imagenes
0,AX_T2_FLAIR,2011-02-01,AXIAL,T2,2D,20
1,AX_T2_FLAIR,NaN,AXIAL,T2,2D,1
2,sag_3D_FSPGR_BRAVO_straight,2011-02-01,SAGITAL,T1,3D,148
3,sag_3D_FSPGR_BRAVO_straight,NaN,SAGITAL,T1,3D,1
4,NaN,NaN,AXIAL,T2,2D,1
5,NaN,NaN,SAGITAL,T1,3D,1


Extracted 0 DICOM file(s) in: /content/dicom_temp/3000
No local DICOM files to display.

--- 3. EXOME ---
VCF extracted to: /content/vcf_temp/3000/PPMI_SI_3000.raw.vcf
VCF SUMMARY — PATNO 3000
Variants read:             199,887
Chromosomes present:       chr1, chrM
Is VCF annotated (ANN/CSQ)?: False
Relevant PD genes:         N/A (VCF without genomic annotation)


## 5. Temporary File Cleanup

The `.dcm` and `.vcf` files extracted on demand are stored in `/content/dicom_temp` and `/content/vcf_temp` (Colab's ephemeral disk, not in Drive). Run this cell to free them when you finish reviewing a patient.

In [15]:
import shutil

def limpiar_temporales(patno=None):
    """If patno is specified, only its folder is deleted. If not, everything extracted is deleted."""
    objetivos = []
    if patno is not None:
        objetivos = [
            os.path.join(DICOM_TEMP_DIR, str(int(patno))),
            os.path.join(VCF_TEMP_DIR, str(int(patno))),
        ]
    else:
        objetivos = [DICOM_TEMP_DIR, VCF_TEMP_DIR]

    for carpeta in objetivos:
        if os.path.exists(carpeta):
            shutil.rmtree(carpeta)
            os.makedirs(carpeta, exist_ok=True)
            print("Cleaned:", carpeta)

# limpiar_temporales(PATNO_A_EXPLORAR)  # uncomment to clean only this patient
# limpiar_temporales()                  # uncomment to clean everything